## Классификация и регрессия

Задание. Представленный датасет использовать для решения задачи регрессии и задачи классификации.
1. Обучить обе модели.
2. Оценить качество обучения.
3. Провести инференс на 3 примерах.
4. Определить лучшую модель.

In [1]:
import torch
import random
import numpy as np

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True

In [2]:
#dataset
import pandas as pd
df = pd.read_csv('winequality.csv',sep=';')

Информация о наборе данных

представлены данные о качестве напитков "Vinho Verde" (  Modeling wine preferences by data mining from physicochemical properties // P. Cortez, A. Cerdeira, Fernando Almeida, Telmo Matos, J. Reis. 2009).  Из-за конфиденциальности доступны только физико-химические (входные данные) и органолептические (выходные данные) переменные (например, нет данных о сортах винограда, марке вина, отпускной цене вина и т.д.).

Эти наборы данных можно рассматривать как для задачи **классификации**, так и **регрессии**.  Классы не сбалансированы (обычных вин гораздо больше, чем высококачественных или некачественных). 

In [3]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


Входные переменные (на основе физико-химического анализа):
1. - fixed acidity
2. - volatile acidity
3. - citric acid
4. - residual sugar
5. - chlorides
6. - free sulfur dioxide
7. - total sulfur dioxide
8. - density
9. - pH
10. - sulphates
11. - alcohol

Целевая переменная (на основе органолептического анализа): 

12. - quality (значение между 0 и 10)

In [5]:
df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


Сформировать обучающую и контрольную выборки

In [6]:
X = df.drop(columns=['quality']).values
Y = (df['quality'].values - df['quality'].min()).astype(int)  # 0-indexed для классификации (3->0, ..., 8->5)

Количество класов

In [7]:
n_classes = len(np.unique(Y))
print('Количество классов:', n_classes)

Количество классов: 6


In [8]:
#train test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, shuffle=True)


In [9]:
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

Прежде чем переходить к построению модели, вычислите BaseRate.

Ваша модель должна быть лучше Zero Rule. 

- Вы можете сравнить свою модель с теоретической оценкой baseline или случайного угадывания и использовать ее для  оценки  вашей модели.

- Для задачи классификации можно использовать sklearn.dummy.DummyClassifier, который предлагает автоматизированное решение для следующих базовых стратегий: “most_frequent”, “prior”, “stratified”, “uniform”, “constant”. Например, 
  -“most_frequent” всегда возвращает наиболее часто встречающуюся метку класса.
 
 При оценке точности для задач несбалансированной классификации стоит использовать AUC.

Перед построением модели определите свой baseline  и установите правила, по которым вы будете оценивать свою 
окончательную модель.

Zero Rule Algorithm для задачи классификации

In [10]:
from random import randrange
 
def zero_rule_algorithm_classification(train, test):
 output_values = [row[-1] for row in train]
 prediction = max(set(output_values), key=output_values.count)
 predicted = [prediction for i in range(len(test))]
 return predicted

random.seed = 42

Построение модели

In [11]:
class ClassifierNet(torch.nn.Module):
    def __init__(self, n_hidden_neurons):
        super(ClassifierNet, self).__init__()
        self.fc1 = torch.nn.Linear(11, n_hidden_neurons)
        self.fc2 = torch.nn.Linear(n_hidden_neurons, n_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x

    def inference(self, x):
        logits = self.forward(x)
        return torch.softmax(logits, dim=1)

class_net = ClassifierNet(n_hidden_neurons=32)

In [12]:
loss = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(class_net.parameters(), lr=0.01)

Обучение

In [13]:
#здесь нужна  корректировка индексов классов
batch_size = 32

for epoch in range(2000):
    order = np.random.permutation(len(X_train))
    for start_index in range(0, len(X_train), batch_size):
        optimizer.zero_grad()
        
        batch_indexes = order[start_index:start_index+batch_size]
        
        x_batch = X_train[batch_indexes]
        y_batch = y_train[batch_indexes]
        
        preds = class_net.forward(x_batch)
        
        loss_value = loss(preds, y_batch)
        loss_value.backward()
        
        optimizer.step()
        
    if epoch % 100 == 0:
        test_preds = class_net.forward(X_test)
        test_preds = test_preds.argmax(dim=1)
        print((test_preds == y_test ).float().mean())

tensor(0.4833)
tensor(0.5917)
tensor(0.6146)
tensor(0.5604)
tensor(0.6208)
tensor(0.5854)
tensor(0.6292)
tensor(0.5854)
tensor(0.6083)
tensor(0.6250)
tensor(0.6250)
tensor(0.6042)
tensor(0.5979)
tensor(0.5979)
tensor(0.6187)
tensor(0.6125)
tensor(0.6104)
tensor(0.6208)
tensor(0.5792)
tensor(0.6021)


# Регрессия

Задачи регрессии требуют предсказания реального значения.

Базовой стратегией здесь является предсказание центральной тенденции. Это может быть среднее значение или медиана.
Использование среднего значения будет давать меньшую ошибку, чем случайное предсказание выходного значения.

Zero Rule Algorithm для задачи регрессии

In [14]:
from random import randrange
 
def zero_rule_algorithm_regression(train, test):
 output_values = [row[-1] for row in train]
 prediction = sum(output_values) / float(len(output_values))
 predicted = [prediction for i in range(len(test))]
 return predicted

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, shuffle=True)

In [16]:
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.FloatTensor(y_train)
y_test = torch.FloatTensor(y_test)

In [17]:
X_train.unsqueeze_(1)
X_test.unsqueeze_(1)

Построение модели

In [18]:
class RegressorNet(torch.nn.Module):
    def __init__(self, n_hidden_neurons):
        super(RegressorNet, self).__init__()
        self.fc1 = torch.nn.Linear(11, n_hidden_neurons)
        self.fc2 = torch.nn.Linear(n_hidden_neurons, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x

    def inference(self, x):
        return self.forward(x).squeeze(-1)

reg_net = RegressorNet(n_hidden_neurons=32)

In [19]:
loss = torch.nn.MSELoss()
optimizer = torch.optim.Adam(reg_net.parameters(), lr=1.0e-3)

In [20]:
y_train.shape

torch.Size([1119])

Обучение

In [21]:
batch_size = 32

for epoch in range(5000):
    order = np.random.permutation(len(X_train))
    for start_index in range(0, len(X_train), batch_size):
        optimizer.zero_grad()
        
        batch_indexes = order[start_index:start_index+batch_size]
        
        x_batch = X_train[batch_indexes]
        y_batch = y_train[batch_indexes]
        
        preds = reg_net.forward(x_batch)
        
        loss_value = loss(preds, y_batch)
        loss_value.backward()
        
        optimizer.step()
        
    if epoch % 100 == 0:
        test_preds = reg_net.forward(X_test)
        
        print(loss(test_preds, y_test).float())

/Users/bereznevn/PyCharmMiscProject/.venv/lib/python3.9/site-packages/torch/nn/modules/loss.py:616: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Users/bereznevn/PyCharmMiscProject/.venv/lib/python3.9/site-packages/torch/nn/modules/loss.py:616: UserWarning: Using a target size (torch.Size([31])) that is different to the input size (torch.Size([31, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/Users/bereznevn/PyCharmMiscProject/.venv/lib/python3.9/site-packages/torch/nn/modules/loss.py:616: UserWarning: Using a target size (torch.Size([480])) that is different to the input size (torch.Size([480, 1])). This will likely lead to incorrect re

tensor(4.7368, grad_fn=<MseLossBackward0>)
tensor(0.7058, grad_fn=<MseLossBackward0>)
tensor(0.7024, grad_fn=<MseLossBackward0>)
tensor(0.6937, grad_fn=<MseLossBackward0>)
tensor(0.6802, grad_fn=<MseLossBackward0>)
tensor(0.7208, grad_fn=<MseLossBackward0>)
tensor(0.6822, grad_fn=<MseLossBackward0>)
tensor(0.6918, grad_fn=<MseLossBackward0>)
tensor(0.6960, grad_fn=<MseLossBackward0>)
tensor(0.6823, grad_fn=<MseLossBackward0>)
tensor(0.7062, grad_fn=<MseLossBackward0>)
tensor(0.6833, grad_fn=<MseLossBackward0>)
tensor(0.6789, grad_fn=<MseLossBackward0>)
tensor(0.6833, grad_fn=<MseLossBackward0>)
tensor(0.6790, grad_fn=<MseLossBackward0>)
tensor(0.6780, grad_fn=<MseLossBackward0>)
tensor(0.6781, grad_fn=<MseLossBackward0>)
tensor(0.6825, grad_fn=<MseLossBackward0>)
tensor(0.6876, grad_fn=<MseLossBackward0>)
tensor(0.6773, grad_fn=<MseLossBackward0>)
tensor(0.6987, grad_fn=<MseLossBackward0>)
tensor(0.6776, grad_fn=<MseLossBackward0>)
tensor(0.6785, grad_fn=<MseLossBackward0>)
tensor(0.68

Инференс

In [ ]:
idx = np.random.randint(len(X_test))
input = X_test[idx]

In [ ]:
output = reg_net.forward(input)

In [ ]:
loss(y_test[idx], output)

In [ ]:
print("NET output = ",np.round(output.detach().numpy()) )
print("Ground Truth = ", y_test[idx].numpy()  )


## Метрики

Оценить качество моделей.
Сравнить оба решения.

In [ ]:
# Оценка классификатора (тестовая выборка — новый сплит с фиксированным seed)
from sklearn.model_selection import train_test_split
_, X_test_cls, _, y_test_cls = train_test_split(X, Y, test_size=0.3, shuffle=True, random_state=0)
X_test_cls = torch.FloatTensor(X_test_cls)
y_test_cls = torch.LongTensor(y_test_cls)

with torch.no_grad():
    preds_cls = class_net(X_test_cls).argmax(dim=1)
    acc = (preds_cls == y_test_cls).float().mean().item()
print('Классификатор: accuracy =', round(acc, 4))

# Оценка регрессора (текущие X_test, y_test — из блока регрессии)
with torch.no_grad():
    preds_reg = reg_net(X_test).squeeze()
    mse = loss(preds_reg, y_test).item()
    mae = (preds_reg - y_test).abs().mean().item()
print('Регрессор: MSE =', round(mse, 4), ', MAE =', round(mae, 4))

In [ ]:
# Сравнение: точность по классам (регрессор округляем до целого класса 0..5)
with torch.no_grad():
    pred_reg_class = torch.clamp(torch.round(reg_net(X_test_cls).squeeze()), 0, n_classes - 1).long()
    acc_reg_as_class = (pred_reg_class == y_test_cls).float().mean().item()
print('Регрессор как классификатор (округление выхода): accuracy =', round(acc_reg_as_class, 4))
print('Лучшая модель по accuracy на метках классов:', 'Классификатор' if acc >= acc_reg_as_class else 'Регрессор (округлённый)')